In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import spacy
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import TextVectorization
from collections import Counter
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
df_clean = pd.read_csv("./asset/sentiment140_cleaned.csv")

In [5]:
df_clean

,target,text,text_clean,text_nlp
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",awww thats a bummer you shoulda got david carr...,awww bummer shoulda get david carr day
1,0,is upset that he can't update his Facebook by ...,is upset that he cant update his facebook by t...,upset not update facebook texte cry result sch...
2,0,@Kenichan I dived many times for the ball. Man...,i dived many times for the ball managed to sav...,dive time ball manage save rest bound
3,0,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire,body feel itchy like fire
4,0,"@nationwideclass no, it's not behaving at all....",no its not behaving at all im mad why am i her...,behave mad not
...,...,...,...,...
1587843,1,Just woke up. Having no school is the best fee...,just woke up having no school is the best feel...,wake have school good feeling
1587844,1,TheWDB.com - Very cool to hear old Walt interv...,thewdbcom very cool to hear old walt interviews,thewdbcom cool hear old walt interview
1587845,1,Are you ready for your MoJo Makeover? Ask me f...,are you ready for your mojo makeover ask me fo...,ready mojo makeover ask detail
1587846,1,Happy 38th Birthday to my boo of alll time!!! ...,happy th birthday to my boo of alll time tupac...,happy th birthday boo alll time tupac amaru sh...


## TRAIN/TEST

In [17]:
df_clean = df_clean.dropna(subset=['text_nlp']).reset_index(drop=True)

X = df_clean['text_nlp']
y = df_clean['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

Train : 1,270,277 | Test : 317,570


## Tokenisation

In [18]:
VOCAB_SIZE = 50000
MAX_LEN = 30
BATCH_SIZE = 200_000

counter = Counter()
for i in range(0, len(X_train), BATCH_SIZE):
    batch = X_train[i : i + BATCH_SIZE]
    for text in batch:
        counter.update(text.split())
    print(f"Batch {i // BATCH_SIZE + 1} compté")

vocab = ['', '[UNK]'] + [word for word, _ in counter.most_common(VOCAB_SIZE - 2)]

vectorizer = TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAX_LEN)
vectorizer.set_vocabulary(vocab)
print("Terminé !")

Batch 1 compté
Batch 2 compté
Batch 3 compté
Batch 4 compté
Batch 5 compté
Batch 6 compté
Batch 7 compté
Terminé !


In [19]:
def vectorize_in_batches(data, vectorizer, batch_size):
    results = []
    total = len(data)
    
    for i in range(0, total, batch_size):
        batch = data[i : i + batch_size]
        vectorized = vectorizer(batch)
        results.append(vectorized.numpy())
        print(f"Batch {i // batch_size + 1}/{-(-total // batch_size)} traité — {min(i + batch_size, total):,} / {total:,}")
    
    return np.concatenate(results, axis=0)

X_train_vec = vectorize_in_batches(X_train, vectorizer, BATCH_SIZE)
X_test_vec  = vectorize_in_batches(X_test,  vectorizer, BATCH_SIZE)

print(f"X_train_vec shape : {X_train_vec.shape}")
print(f"X_test_vec shape  : {X_test_vec.shape}")

Batch 1/7 traité — 200,000 / 1,270,277
Batch 2/7 traité — 400,000 / 1,270,277
Batch 3/7 traité — 600,000 / 1,270,277
Batch 4/7 traité — 800,000 / 1,270,277
Batch 5/7 traité — 1,000,000 / 1,270,277
Batch 6/7 traité — 1,200,000 / 1,270,277
Batch 7/7 traité — 1,270,277 / 1,270,277
Batch 1/2 traité — 200,000 / 317,570
Batch 2/2 traité — 317,570 / 317,570
X_train_vec shape : (1270277, 30)
X_test_vec shape  : (317570, 30)


In [20]:
# Vérifier qu'il n'y a pas de séquence de longueur différente de 30
print(f"Toutes les séquences font bien 30 tokens : {X_train_vec.shape[1] == 30}")
# Voir un exemple de padding
print(X_train_vec[0])  # tu verras des 0 en fin de séquence si le tweet est court

Toutes les séquences font bien 30 tokens : True
[  70    1  968    3    1   55  148    1 1170  176    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]


In [21]:
EMBEDDING_DIM = 64   # dimension des vecteurs d'embedding

model = Sequential([
    # Couche Embedding : transforme chaque token (entier) en vecteur dense
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),

    # LTSM Bidirectionnel : lit la séquence dans les deux sens
    Bidirectional(LSTM(64, return_sequences=False)),

    # Dropout : désactive aléatoirement 40% des neurones → évite l'overfitting
    Dropout(0.4),

    # Couche Dense intérmediaire
    Dense(31, activation='relu'),

    Dropout(0.2),
    
    # Couche de sortie : 1 neurone, sigmoid → probabilité entre 0 et 1
    Dense(1, activation='sigmoid')
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # classification binaire
    metrics=['accuracy']
)
model.build(input_shape=(None, MAX_LEN))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 30, 64)         │     3,200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 31)             │         3,999 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 31)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            32 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,270,079 (12.47 MB)

 Trainable params: 3,270,079 (12.47 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

model.fit(
    X_train_vec, y_train,
    shuffle=True, 
    epochs=30,
    validation_split=0.1,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
   1/2233 ━━━━━━━━━━━━━━━━━━━━ 4:21 117ms/step - accuracy: 0.7988 - loss: 0.4637

W0000 00:00:1776262197.996409    3983 cpu_allocator_impl.cc:82] Allocation of 274379760 exceeds 10% of free system memory.


2233/2233 ━━━━━━━━━━━━━━━━━━━━ 165s 74ms/step - accuracy: 0.7927 - loss: 0.4450 - val_accuracy: 0.7839 - val_loss: 0.4574
Epoch 2/30
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 166s 74ms/step - accuracy: 0.8041 - loss: 0.4249 - val_accuracy: 0.7836 - val_loss: 0.4627
Epoch 3/30
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 156s 70ms/step - accuracy: 0.8140 - loss: 0.4039 - val_accuracy: 0.7802 - val_loss: 0.4845


## Test avant export

In [28]:
# Evaluation sur le jeu de test
loss, accuracy = model.evaluate(X_test_vec, y_test, verbose=1)

print(f"Loss     : {loss:.4f}")
print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")

9925/9925 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.7839 - loss: 0.4573
Loss     : 0.4573
Accuracy : 0.7839 (78.39%)


In [30]:
# Rapport de classification détaillé
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (model.predict(X_test_vec) > 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=['Négatif', 'Positif']))

9925/9925 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step
              precision    recall  f1-score   support

     Négatif       0.79      0.78      0.78    158880
     Positif       0.78      0.79      0.79    158690

    accuracy                           0.78    317570
   macro avg       0.78      0.78      0.78    317570
weighted avg       0.78      0.78      0.78    317570



In [31]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
print("Matrice de confusion :")
print(cm)

Matrice de confusion :
[[123505  35375]
 [ 33241 125449]]


# Save model

In [32]:
model.save('sentiment_model_v2.keras')
print("Modèle sauvegardé !")

Modèle sauvegardé !


In [33]:
# Sauvegarde du vectorizer (indispensable pour la production)
import pickle
with open('vectorizer_v2.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print("Vectorizer sauvegardé !")

Vectorizer sauvegardé !
